# Story Synopsis Generation Notebook

This notebook generates one-sentence plot synopses for all human and AI story responses.

The goal is to support a narrative-level story crowding kernel:

$K^{plot}_k(x,y)=\frac{1+\cos(f(s(x)), f(s(y)))}{2}$

where $s(x)$ is a deterministic one-sentence plot synopsis of story $x$.

We generate the synopses once, save them to disk, and later load them in the main crowding-analysis notebook.

## 1. Imports and paths

We load standardized story data from the previous analysis outputs and AI processed files. We save all batch inputs, manifests, raw outputs, parsed synopses, and final merged files under `analysis_outputs/story_synopses/`.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Any, List, Optional
import os
import re
import json
import time
import hashlib
import datetime as dt

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)

# load_dotenv()

# assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"

client = OpenAI(api_key="")

BASE_DIR = Path(".")
ANALYSIS_DIR = Path("analysis_outputs")
STANDARDIZED_DIR = ANALYSIS_DIR / "standardized_loaded_data"
AI_PROCESSED_DIR = Path("ai_data/processed")

SYNOPSIS_DIR = ANALYSIS_DIR / "story_synopses"
BATCH_DIR = SYNOPSIS_DIR / "openai_batches"
RAW_DIR = SYNOPSIS_DIR / "raw_outputs"
PARSED_DIR = SYNOPSIS_DIR / "parsed"
FINAL_DIR = SYNOPSIS_DIR / "final"

for p in [SYNOPSIS_DIR, BATCH_DIR, RAW_DIR, PARSED_DIR, FINAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OPENAI_SYNOPSIS_MODEL = "gpt-5.4"

## 2. Utility functions

These helpers handle JSONL files, stable IDs, and robust text extraction from the OpenAI Responses API output.

In [2]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def stable_hash(obj: Any) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def short_hash(obj: Any, n: int = 16) -> str:
    return stable_hash(obj)[:n]


def write_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    path = Path(path)
    if not path.exists():
        return []
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out


def extract_text_from_responses_api_body(body: dict) -> str:
    """
    Extract visible text from an OpenAI Responses API response body.
    """
    if not isinstance(body, dict):
        return ""

    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def rough_word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b[\w']+\b", text))


def clean_synopsis_text(text: str) -> str:
    """
    Light cleaning for model-produced synopses.
    Keeps content intact but removes common wrappers.
    """
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = re.sub(r"^['\"“”‘’]+|['\"“”‘’]+$", "", text).strip()

    # Remove an accidental label if present.
    text = re.sub(r"^(synopsis|plot synopsis)\s*:\s*", "", text, flags=re.I).strip()

    # Collapse whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text

## 3. Load story data to summarize

We want synopses for:

1. human story responses;
2. AI main story responses;
3. AI robustness story responses, including temperature and personality conditions.

The safest source for AI robustness is the provider-specific `story_generations_success_only.pkl` files, because those include all successful story generations.

In [4]:
# Human stories from standardized analysis data.
human_standard_df = pd.read_pickle(STANDARDIZED_DIR / "human_standard_all_tasks.pkl")
human_story_df = human_standard_df.query("task_family == 'story'").copy()

print("Human story rows:", human_story_df.shape)
display(
    human_story_df
    .groupby("condition_id")
    .agg(n=("response_text", "size"), n_unique=("response_text", "nunique"))
    .reset_index()
)

# AI story files with main + robustness success-only generations.
AI_STORY_FILES = [
    ("openai", "gpt-5.4", AI_PROCESSED_DIR / "openai__gpt-5.4__story_generations_all_scenarios.pkl"),
    ("anthropic", "claude-sonnet-4-5", AI_PROCESSED_DIR / "anthropic__claude-sonnet-4-5__story_generations_success_only.pkl"),
    ("gemini", "gemini-2.5-flash", AI_PROCESSED_DIR / "gemini__gemini-2.5-flash__story_generations_success_only.pkl"),
]

ai_story_dfs = []

for provider, model, path in AI_STORY_FILES:
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_pickle(path).copy()
    df["provider"] = provider
    df["model"] = model
    df["task_family"] = "story"
    df["source_type"] = "ai"

    # Standardize response text.
    df["response_text"] = df["text"].astype(str)

    # Standardize scenario name.
    if "analysis_scenario_name" not in df.columns:
        df["analysis_scenario_name"] = df["scenario_name"]

    df["analysis_scenario_name"] = df["analysis_scenario_name"].fillna(df["scenario_name"])

    ai_story_dfs.append(df)

    print(f"Loaded {provider} / {model}: {df.shape} from {path}")

ai_story_df = pd.concat(ai_story_dfs, ignore_index=True, sort=False)

print("AI story rows:", ai_story_df.shape)
display(
    ai_story_df
    .groupby(["provider", "model", "analysis_scenario_name", "temperature"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["provider", "analysis_scenario_name", "temperature"])
)

Human story rows: (87, 17)


,condition_id,n,n_unique
0,10491,35,35
1,93742,32,32
2,93855,20,20


Loaded openai / gpt-5.4: (3090, 28) from ai_data/processed/openai__gpt-5.4__story_generations_all_scenarios.pkl
Loaded anthropic / claude-sonnet-4-5: (3090, 30) from ai_data/processed/anthropic__claude-sonnet-4-5__story_generations_success_only.pkl
Loaded gemini / gemini-2.5-flash: (3090, 31) from ai_data/processed/gemini__gemini-2.5-flash__story_generations_success_only.pkl
AI story rows: (9270, 33)


,provider,model,analysis_scenario_name,temperature,n
0,anthropic,claude-sonnet-4-5,neutral_main_t1,1.0,150
1,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.3,30
2,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,0.7,30
3,anthropic,claude-sonnet-4-5,personality_grid,0.3,960
4,anthropic,claude-sonnet-4-5,personality_grid,0.7,960
5,anthropic,claude-sonnet-4-5,personality_grid,1.0,960
6,gemini,gemini-2.5-flash,neutral_main_t1,1.0,150
7,gemini,gemini-2.5-flash,neutral_temperature_robustness,0.7,30
8,gemini,gemini-2.5-flash,neutral_temperature_robustness,1.3,30
9,gemini,gemini-2.5-flash,personality_grid,0.7,960


## 4. Standardize story rows

We create one standardized table with one row per story response. Each row receives a stable `story_uid`.

The `story_uid` is based on source metadata and exact story text, so it remains stable across notebook runs.

In [6]:
def normalize_condition_id(x) -> str:
    if pd.isna(x):
        return ""
    try:
        f = float(x)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return str(x)


def get_story_prompt_from_ai_row(row: pd.Series) -> str:
    """
    Best-effort extraction of story prompt text from AI generation rows.
    """
    for col in ["prompt", "story_prompt", "condition_label"]:
        val = row.get(col)
        if isinstance(val, str) and val.strip() and val.strip().lower() != "nan":
            return val.strip()

    user_prompt = row.get("user_prompt")
    if isinstance(user_prompt, str) and user_prompt.strip():
        # The generation notebook commonly stored a full user prompt.
        # Keep it as metadata if no cleaner prompt field exists.
        return user_prompt.strip()

    return ""


def standardize_human_story_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["source_type"] = "human"
    out["provider"] = "human"
    out["model"] = "human"
    out["analysis_scenario_name"] = "human"
    out["scenario_name"] = "human"
    out["temperature"] = np.nan
    out["persona_id"] = np.nan

    out["condition_id"] = out["condition_id"].map(normalize_condition_id)
    out["story_prompt"] = out["condition_label"].astype(str)
    out["story_text"] = out["response_text"].astype(str)

    if "participant_id" not in out.columns:
        out["participant_id"] = np.nan

    keep = [
        "source_type",
        "provider",
        "model",
        "analysis_scenario_name",
        "scenario_name",
        "temperature",
        "persona_id",
        "condition_id",
        "story_prompt",
        "participant_id",
        "story_text",
    ]

    return out[keep].copy()


def standardize_ai_story_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Repair condition ID for story rows if needed.
    if "condition_id" not in out.columns:
        out["condition_id"] = np.nan

    out["condition_id"] = out["condition_id"].map(normalize_condition_id)

    if "prompt_id" in out.columns:
        prompt_ids = out["prompt_id"].map(normalize_condition_id)
        missing = out["condition_id"].isin(["", "nan", "None"])
        out.loc[missing, "condition_id"] = prompt_ids.loc[missing]

    out["story_prompt"] = out.apply(get_story_prompt_from_ai_row, axis=1)
    out["story_text"] = out["response_text"].astype(str)

    if "participant_id" not in out.columns:
        out["participant_id"] = np.nan
    if "persona_id" not in out.columns:
        out["persona_id"] = np.nan
    if "scenario_name" not in out.columns:
        out["scenario_name"] = out["analysis_scenario_name"]

    keep = [
        "source_type",
        "provider",
        "model",
        "analysis_scenario_name",
        "scenario_name",
        "temperature",
        "persona_id",
        "condition_id",
        "story_prompt",
        "participant_id",
        "story_text",
    ]

    return out[keep].copy()


human_story_standard = standardize_human_story_rows(human_story_df)
ai_story_standard = standardize_ai_story_rows(ai_story_df)

story_all_df = pd.concat(
    [human_story_standard, ai_story_standard],
    ignore_index=True,
    sort=False,
)

story_all_df["story_text"] = story_all_df["story_text"].fillna("").astype(str)
story_all_df = story_all_df[story_all_df["story_text"].str.strip().ne("")].copy()

# # Stable ID for each story row.
# story_all_df["story_uid"] = story_all_df.apply(
#     lambda row: short_hash(
#         {
#             "source_type": row["source_type"],
#             "provider": row["provider"],
#             "model": row["model"],
#             "analysis_scenario_name": row["analysis_scenario_name"],
#             "scenario_name": row["scenario_name"],
#             "temperature": None if pd.isna(row["temperature"]) else float(row["temperature"]),
#             "persona_id": None if pd.isna(row["persona_id"]) else str(row["persona_id"]),
#             "condition_id": row["condition_id"],
#             "participant_id": None if pd.isna(row["participant_id"]) else str(row["participant_id"]),
#             "story_text": row["story_text"],
#         },
#         n=24,
#     ),
#     axis=1,
# )

# # Diagnostics.
# story_all_df["story_word_count"] = story_all_df["story_text"].map(rough_word_count)

# print("All story rows:", story_all_df.shape)
# print("Unique story_uid:", story_all_df["story_uid"].nunique())

# assert story_all_df["story_uid"].is_unique, "story_uid collision or duplicate exact metadata rows detected."
# Reset row order and create a row-level position.
# This makes story_uid unique even if two rows have identical metadata and identical story text.
story_all_df = story_all_df.reset_index(drop=True)
story_all_df["story_row_idx"] = np.arange(len(story_all_df), dtype=int)

# Stable row-level ID for each story row.
# We include story_row_idx to distinguish exact duplicate rows.
# The text-level synopsis ID is still handled later by story_text_uid, so duplicate texts
# will still be summarized only once.
story_all_df["story_uid"] = story_all_df.apply(
    lambda row: short_hash(
        {
            "story_row_idx": int(row["story_row_idx"]),
            "source_type": row["source_type"],
            "provider": row["provider"],
            "model": row["model"],
            "analysis_scenario_name": row["analysis_scenario_name"],
            "scenario_name": row["scenario_name"],
            "temperature": None if pd.isna(row["temperature"]) else float(row["temperature"]),
            "persona_id": None if pd.isna(row["persona_id"]) else str(row["persona_id"]),
            "condition_id": row["condition_id"],
            "participant_id": None if pd.isna(row["participant_id"]) else str(row["participant_id"]),
            "story_text": row["story_text"],
        },
        n=24,
    ),
    axis=1,
)

# Diagnostics.
story_all_df["story_word_count"] = story_all_df["story_text"].map(rough_word_count)

print("All story rows:", story_all_df.shape)
print("Unique story_uid:", story_all_df["story_uid"].nunique())

assert story_all_df["story_uid"].is_unique, "story_uid should be row-level unique."
display(
    story_all_df
    .groupby(["source_type", "provider", "model", "analysis_scenario_name"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["source_type", "provider", "analysis_scenario_name"])
)

story_all_df.head()

All story rows: (9357, 14)
Unique story_uid: 9357


,source_type,provider,model,analysis_scenario_name,n
0,ai,anthropic,claude-sonnet-4-5,neutral_main_t1,150
1,ai,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,60
2,ai,anthropic,claude-sonnet-4-5,personality_grid,2880
3,ai,gemini,gemini-2.5-flash,neutral_main_t1,150
4,ai,gemini,gemini-2.5-flash,neutral_temperature_robustness,60
5,ai,gemini,gemini-2.5-flash,personality_grid,2880
6,ai,openai,gpt-5.4,neutral_main_t1,150
7,ai,openai,gpt-5.4,neutral_temperature_robustness,60
8,ai,openai,gpt-5.4,personality_grid,2880
9,human,human,human,human,87


,source_type,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,condition_id,story_prompt,participant_id,story_text,story_row_idx,story_uid,story_word_count
0,human,human,human,human,human,NaN,NaN,10491,A short Horror story . Something to chill the bones in one hundred words or less .,10880,"I was running . Tree branches whipped past my face , and I felt the hot warmth of my blood mix with the cold sweat . Footsteps behind me continued to keep pace , no matter how ...",0,f4a9d7a3b0e9b00d78193112,116
1,human,human,human,human,human,NaN,NaN,10491,A short Horror story . Something to chill the bones in one hundred words or less .,11118,"I breathed heavily under the covers . A creak made me jump unwillingly . Dad was n't here , so he did n't check the closet . What if *it* were there ? What if it was waiting fo...",1,2f4a43b282f6c3388f992cfe,111
2,human,human,human,human,human,NaN,NaN,10491,A short Horror story . Something to chill the bones in one hundred words or less .,15787,"I pound my fists against the old wooden door within the creaking old house I found in the woods . The wind had blown it shut , I told myself , no need to worry . This was an ho...",2,8c9a5d5a6cef6db6cbad6597,133
3,human,human,human,human,human,NaN,NaN,10491,A short Horror story . Something to chill the bones in one hundred words or less .,28897,The town square sparkled like the 4th of July sky . Children 's laughter filled the air ; old friends were catching up . Hearing them made me think of Junior and my wife and my...,3,c07604df10908a34465f4d69,125
4,human,human,human,human,human,NaN,NaN,10491,A short Horror story . Something to chill the bones in one hundred words or less .,32611,"It had been a year since the accident , and not one day passed I did n't think of Lindsay . The fiery wreck off Intersate 15 that took my right hand and burned 40 % of my body ...",4,724837d9bd9001362d6c5350,241


In [7]:
dupe_cols = [
    "source_type",
    "provider",
    "model",
    "analysis_scenario_name",
    "scenario_name",
    "temperature",
    "persona_id",
    "condition_id",
    "participant_id",
    "story_text",
]

dupes = story_all_df[
    story_all_df.duplicated(dupe_cols, keep=False)
].copy()

print("Duplicate exact metadata/text rows:", len(dupes))
display(dupes[dupe_cols].head(10))

Duplicate exact metadata/text rows: 2


,source_type,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,condition_id,participant_id,story_text
3525,ai,anthropic,claude-sonnet-4-5,personality_grid,anthropic_personality_grid_t03,0.3,introverted__agreeable__unconscientious__neurotic__closed_to_experience,10491,NaN,"**The Visitor**\n\nI never answer the door anymore. Not since that night.\n\nThe knocking started at 3 AM—slow, deliberate. I stayed in bed, covers pulled tight. It continued. ..."
3864,ai,anthropic,claude-sonnet-4-5,personality_grid,anthropic_personality_grid_t03,0.3,introverted__agreeable__unconscientious__neurotic__closed_to_experience,10491,NaN,"**The Visitor**\n\nI never answer the door anymore. Not since that night.\n\nThe knocking started at 3 AM—slow, deliberate. I stayed in bed, covers pulled tight. It continued. ..."


In [8]:
# Audit whether exact duplicate story rows are true repeated generations
# or accidental duplicated records.

cols_to_show = [
    "source_type",
    "provider",
    "model",
    "analysis_scenario_name",
    "scenario_name",
    "temperature",
    "persona_id",
    "condition_id",
    "participant_id",
    "story_text",
]

optional_cols = [
    "request_key",
    "custom_id",
    "batch_custom_id",
    "run_idx",
    "source_file",
    "created_at_utc",
]

available_optional_cols = [c for c in optional_cols if c in story_all_df.columns]

dupe_exact = story_all_df[
    story_all_df.duplicated(cols_to_show, keep=False)
].copy()

print("Exact duplicate metadata/text rows:", len(dupe_exact))
display(dupe_exact[cols_to_show + available_optional_cols])

Exact duplicate metadata/text rows: 2


,source_type,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,condition_id,participant_id,story_text
3525,ai,anthropic,claude-sonnet-4-5,personality_grid,anthropic_personality_grid_t03,0.3,introverted__agreeable__unconscientious__neurotic__closed_to_experience,10491,NaN,"**The Visitor**\n\nI never answer the door anymore. Not since that night.\n\nThe knocking started at 3 AM—slow, deliberate. I stayed in bed, covers pulled tight. It continued. ..."
3864,ai,anthropic,claude-sonnet-4-5,personality_grid,anthropic_personality_grid_t03,0.3,introverted__agreeable__unconscientious__neurotic__closed_to_experience,10491,NaN,"**The Visitor**\n\nI never answer the door anymore. Not since that night.\n\nThe knocking started at 3 AM—slow, deliberate. I stayed in bed, covers pulled tight. It continued. ..."


In [9]:
# Audit duplicate request keys in the raw AI story data.
# A duplicate request_key would suggest accidental duplicated records.

if "request_key" in ai_story_df.columns:
    duplicate_request_keys = (
        ai_story_df
        .groupby(["provider", "model", "request_key"], dropna=False)
        .size()
        .reset_index(name="n")
        .query("n > 1")
        .sort_values("n", ascending=False)
    )

    print("Duplicate AI request_key rows:", len(duplicate_request_keys))
    display(duplicate_request_keys.head(20))
else:
    print("No request_key column found in ai_story_df.")

Duplicate AI request_key rows: 0


,provider,model,request_key,n


In [10]:
# Find exact duplicate AI story texts within the same scenario/persona/temperature/condition.
# These are not necessarily bad; they may be genuine model duplicates.

dupe_group_cols = [
    "provider",
    "model",
    "analysis_scenario_name",
    "scenario_name",
    "temperature",
    "persona_id",
    "condition_id",
    "response_text",
]

available_dupe_group_cols = [c for c in dupe_group_cols if c in ai_story_df.columns]

ai_exact_text_dupes = (
    ai_story_df
    .groupby(available_dupe_group_cols, dropna=False)
    .agg(
        n_rows=("response_text", "size"),
        n_request_keys=("request_key", "nunique") if "request_key" in ai_story_df.columns else ("response_text", "size"),
        run_idxs=("run_idx", lambda x: sorted(x.dropna().unique().tolist()) if "run_idx" in ai_story_df.columns else []),
    )
    .reset_index()
    .query("n_rows > 1")
    .sort_values("n_rows", ascending=False)
)

display(ai_exact_text_dupes.head(20))

,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,response_text,n_rows,n_request_keys,run_idxs
884,anthropic,claude-sonnet-4-5,personality_grid,anthropic_personality_grid_t03,0.3,introverted__agreeable__unconscientious__neurotic__closed_to_experience,"**The Visitor**\n\nI never answer the door anymore. Not since that night.\n\nThe knocking started at 3 AM—slow, deliberate. I stayed in bed, covers pulled tight. It continued. ...",2,2,"[2, 8]"


## 5. Decide whether to summarize unique story texts or every row

For narrative crowding, identical story text should receive the same synopsis. To reduce cost, we summarize each unique story text once, then map the synopsis back to all rows.

This preserves duplicates in the later crowding analysis while avoiding duplicate API calls.

In [11]:
unique_story_text_df = (
    story_all_df[["story_text"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_story_text_df["story_text_uid"] = unique_story_text_df["story_text"].map(
    lambda x: short_hash({"story_text": x}, n=24)
)

unique_story_text_df["story_word_count"] = unique_story_text_df["story_text"].map(rough_word_count)

print("Total story rows:", len(story_all_df))
print("Unique story texts:", len(unique_story_text_df))
print("Duplicate rows saved:", len(story_all_df) - len(unique_story_text_df))

display(unique_story_text_df["story_word_count"].describe(percentiles=[.25, .5, .75, .9, .95, .99]))

unique_story_text_df.head()

Total story rows: 9357
Unique story texts: 9356
Duplicate rows saved: 1


count    9356.000000
mean      104.080376
std        10.274394
min        48.000000
25%        99.000000
50%       103.000000
75%       109.000000
90%       117.000000
95%       120.000000
99%       130.000000
max       268.000000
Name: story_word_count, dtype: float64

,story_text,story_text_uid,story_word_count
0,"I was running . Tree branches whipped past my face , and I felt the hot warmth of my blood mix with the cold sweat . Footsteps behind me continued to keep pace , no matter how ...",23432105eff14111d6b15c34,116
1,"I breathed heavily under the covers . A creak made me jump unwillingly . Dad was n't here , so he did n't check the closet . What if *it* were there ? What if it was waiting fo...",0c36d334900ad801f4c0d215,111
2,"I pound my fists against the old wooden door within the creaking old house I found in the woods . The wind had blown it shut , I told myself , no need to worry . This was an ho...",109fd53c9a1da4b3e91ea992,133
3,The town square sparkled like the 4th of July sky . Children 's laughter filled the air ; old friends were catching up . Hearing them made me think of Junior and my wife and my...,94c1583b6d345dcbe280ae96,125
4,"It had been a year since the accident , and not one day passed I did n't think of Lindsay . The fiery wreck off Intersate 15 that took my right hand and burned 40 % of my body ...",c8298a2af4ce36f3dcd16621,241


## 6. Define the synopsis prompt

We use a fixed deterministic prompt and ask for exactly one sentence.

The synopsis should capture narrative content: protagonist, central situation, conflict, and outcome if present. It should avoid style commentary and avoid quoting the story.

In [12]:
SYNOPSIS_SYSTEM_INSTRUCTIONS = (
    "You convert creative stories into concise plot synopses for research analysis. "
    "Follow the instructions exactly. Do not add commentary."
)


def build_synopsis_user_prompt(story_text: str) -> str:
    return (
        "Summarize the following story as exactly one sentence.\n\n"
        "Focus on the plot: the protagonist or central entity, the main situation, "
        "the central conflict or change, and the outcome if it is present.\n\n"
        "Do not evaluate the writing quality. Do not mention the author. "
        "Do not quote the story. Do not include labels such as 'Synopsis:'. "
        "Return only the one-sentence plot synopsis.\n\n"
        "STORY:\n"
        f"{story_text}"
    )


# Preview one prompt.
print(build_synopsis_user_prompt(unique_story_text_df["story_text"].iloc[0])[:2000])

Summarize the following story as exactly one sentence.

Focus on the plot: the protagonist or central entity, the main situation, the central conflict or change, and the outcome if it is present.

Do not evaluate the writing quality. Do not mention the author. Do not quote the story. Do not include labels such as 'Synopsis:'. Return only the one-sentence plot synopsis.

STORY:
I was running . Tree branches whipped past my face , and I felt the hot warmth of my blood mix with the cold sweat . Footsteps behind me continued to keep pace , no matter how hard I ran . When I slowed , they slowed , when I quickened , they increased . Pausing for a moment to catch my breath , I saw the pale white eyes loom up at me out of the darkness . They were quickly followed by a colorful bowtie and red buttons on a bare chest . <newline> <newline> “ Run , run , run as fast as you can ; I ’ ll always catch you , because I ’ m the Gingerbread Man ! ” <newline> <newline> -- -- -- -- <newline> <newline> [ my

## 7. Check existing synopsis outputs for resume/backfill

This notebook is designed to be resumable. If a previous parsed synopsis file exists, we skip story texts that already have successful synopses.

In [13]:
SYNOPSIS_PARSED_JSONL = PARSED_DIR / f"openai__{OPENAI_SYNOPSIS_MODEL}__story_plot_synopses.jsonl"
SYNOPSIS_FINAL_PKL = FINAL_DIR / f"openai__{OPENAI_SYNOPSIS_MODEL}__story_plot_synopses_merged.pkl"
SYNOPSIS_FINAL_CSV = FINAL_DIR / f"openai__{OPENAI_SYNOPSIS_MODEL}__story_plot_synopses_merged.csv"

existing_synopsis_records = read_jsonl(SYNOPSIS_PARSED_JSONL)

existing_success_text_uids = {
    r["story_text_uid"]
    for r in existing_synopsis_records
    if r.get("status") == "success" and r.get("synopsis")
}

print("Existing successful synopsis records:", len(existing_success_text_uids))

remaining_unique_story_text_df = unique_story_text_df[
    ~unique_story_text_df["story_text_uid"].isin(existing_success_text_uids)
].copy()

print("Remaining unique story texts to summarize:", len(remaining_unique_story_text_df))

Existing successful synopsis records: 0
Remaining unique story texts to summarize: 9356


## 8. Build the OpenAI Batch JSONL file

Each JSONL line is one request to `/v1/responses`.

We use `temperature=0` so the synopsis generation is deterministic as much as the API allows. We also keep `max_output_tokens` small because each output should be exactly one sentence.

In [14]:
def make_openai_synopsis_batch_jsonl(
    story_text_df: pd.DataFrame,
    model: str = OPENAI_SYNOPSIS_MODEL,
    output_dir: Path = BATCH_DIR,
    max_output_tokens: int = 80,
) -> dict:
    """
    Create a batch JSONL file for story synopsis generation.
    """
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"openai__{model}__story_plot_synopses__{timestamp}"

    batch_jsonl_path = output_dir / f"{stem}__batch_input.jsonl"
    plan_csv_path = output_dir / f"{stem}__batch_plan.csv"

    plan_df = story_text_df.copy()
    plan_df["custom_id"] = plan_df["story_text_uid"].map(lambda x: f"syn_{x}")

    records = []

    for _, row in plan_df.iterrows():
        body = {
            "model": model,
            "instructions": SYNOPSIS_SYSTEM_INSTRUCTIONS,
            "input": build_synopsis_user_prompt(row["story_text"]),
            "temperature": 0,
            "max_output_tokens": max_output_tokens,
        }

        request = {
            "custom_id": row["custom_id"],
            "method": "POST",
            "url": "/v1/responses",
            "body": body,
        }

        records.append(request)

    write_jsonl(batch_jsonl_path, records)

    # Save plan without giant prompt body, but keep story text for parsing/merge.
    plan_df.to_csv(plan_csv_path, index=False)

    print("Batch JSONL:", batch_jsonl_path)
    print("Plan CSV:   ", plan_csv_path)
    print("Requests:   ", len(plan_df))

    return {
        "batch_jsonl_path": batch_jsonl_path,
        "plan_csv_path": plan_csv_path,
        "n_requests": len(plan_df),
    }


synopsis_batch_files = make_openai_synopsis_batch_jsonl(
    remaining_unique_story_text_df,
    model=OPENAI_SYNOPSIS_MODEL,
    output_dir=BATCH_DIR,
    max_output_tokens=80,
)

synopsis_batch_files

Batch JSONL: analysis_outputs/story_synopses/openai_batches/openai__gpt-5.4__story_plot_synopses__20260501_010648__batch_input.jsonl
Plan CSV:    analysis_outputs/story_synopses/openai_batches/openai__gpt-5.4__story_plot_synopses__20260501_010648__batch_plan.csv
Requests:    9356


{'batch_jsonl_path': PosixPath('analysis_outputs/story_synopses/openai_batches/openai__gpt-5.4__story_plot_synopses__20260501_010648__batch_input.jsonl'),
 'plan_csv_path': PosixPath('analysis_outputs/story_synopses/openai_batches/openai__gpt-5.4__story_plot_synopses__20260501_010648__batch_plan.csv'),
 'n_requests': 9356}

## 9. Submit the OpenAI batch

Run this only if the previous cell produced `n_requests > 0`.

The batch will usually complete quickly, but OpenAI’s Batch API is asynchronous and can take up to 24 hours.

In [15]:
def submit_openai_batch(batch_jsonl_path: Path, model: str, metadata: Optional[Dict[str, str]] = None) -> dict:
    """
    Upload a JSONL file and submit an OpenAI batch job.
    """
    batch_input_file = client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata=metadata or {},
    )

    batch_info = {
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "model": model,
        "submitted_at_utc": now_iso(),
        "local_input_file": str(batch_jsonl_path),
        "status": batch.status,
    }

    manifest_path = batch_jsonl_path.with_suffix(".batch_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(batch_info, f, indent=2)

    print("Submitted batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


if synopsis_batch_files["n_requests"] == 0:
    print("No new synopsis requests to submit.")
    synopsis_batch_info = None
else:
    synopsis_batch_info = submit_openai_batch(
        batch_jsonl_path=synopsis_batch_files["batch_jsonl_path"],
        model=OPENAI_SYNOPSIS_MODEL,
        metadata={
            "task": "story_plot_synopsis",
            "model": OPENAI_SYNOPSIS_MODEL,
            "plan_csv": str(synopsis_batch_files["plan_csv_path"]),
        },
    )

synopsis_batch_info

Submitted batch:
{
  "batch_id": "batch_69f43503ebf481909bed22115e55e609",
  "input_file_id": "file-UFLhYyqwF8SRLPsuQD3A3a",
  "model": "gpt-5.4",
  "submitted_at_utc": "2026-05-01T05:07:16.464516+00:00",
  "local_input_file": "analysis_outputs/story_synopses/openai_batches/openai__gpt-5.4__story_plot_synopses__20260501_010648__batch_input.jsonl",
  "status": "validating"
}


{'batch_id': 'batch_69f43503ebf481909bed22115e55e609',
 'input_file_id': 'file-UFLhYyqwF8SRLPsuQD3A3a',
 'model': 'gpt-5.4',
 'submitted_at_utc': '2026-05-01T05:07:16.464516+00:00',
 'local_input_file': 'analysis_outputs/story_synopses/openai_batches/openai__gpt-5.4__story_plot_synopses__20260501_010648__batch_input.jsonl',
 'status': 'validating'}

## 10. Check batch status

Rerun this cell until `status='completed'`.

If the batch fails or partially fails, we will inspect the error file and backfill missing records.

In [35]:
def check_openai_batch(batch_id: str):
    batch = client.batches.retrieve(batch_id)
    print(batch)
    return batch


# Paste a batch ID here if resuming in a restarted notebook.
BATCH_ID_TO_CHECK = None

if synopsis_batch_info is not None:
    BATCH_ID_TO_CHECK = synopsis_batch_info["batch_id"]

if BATCH_ID_TO_CHECK is None:
    print("No batch ID set.")
else:
    batch_status = check_openai_batch(BATCH_ID_TO_CHECK)

Batch(id='batch_69f43503ebf481909bed22115e55e609', completion_window='24h', created_at=1777612035, endpoint='/v1/responses', input_file_id='file-UFLhYyqwF8SRLPsuQD3A3a', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1777615034, error_file_id=None, errors=None, expired_at=None, expires_at=1777698435, failed_at=None, finalizing_at=1777613676, in_progress_at=1777612099, metadata={'task': 'story_plot_synopsis', 'model': 'gpt-5.4', 'plan_csv': 'analysis_outputs/story_synopses/openai_batches/openai__gpt-5.4__story_plot_synopses__20260501_010648__batch_plan.csv'}, model='gpt-5.4-2026-03-05', output_file_id='file-MsgZXKcN17vaCeYkfF9Dga', request_counts=BatchRequestCounts(completed=9356, failed=0, total=9356), usage=BatchUsage(input_tokens=2387256, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=409680, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2796936))


## 11. Download batch results

Once the batch is completed, download the output and error JSONL files.

OpenAI batch results are not guaranteed to be returned in the same order as the input file, so we use `custom_id` to join results back to the plan.

In [36]:
def download_openai_batch_results(batch_id: str, output_dir: Path = RAW_DIR):
    """
    Download OpenAI batch output and error files.
    """
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"Batch is not completed yet. Current status: {batch.status}")
        return None, None

    output_path = output_dir / f"{batch_id}__output.jsonl"
    error_path = output_dir / f"{batch_id}__errors.jsonl"

    if batch.output_file_id:
        file_response = client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")

    if batch.error_file_id:
        error_response = client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None
        print("No error file.")

    return output_path, error_path


if BATCH_ID_TO_CHECK is None:
    print("No batch ID set.")
else:
    synopsis_output_path, synopsis_error_path = download_openai_batch_results(
        BATCH_ID_TO_CHECK,
        output_dir=RAW_DIR,
    )

synopsis_output_path if BATCH_ID_TO_CHECK is not None else None

Downloaded output: analysis_outputs/story_synopses/raw_outputs/batch_69f43503ebf481909bed22115e55e609__output.jsonl
No error file.


PosixPath('analysis_outputs/story_synopses/raw_outputs/batch_69f43503ebf481909bed22115e55e609__output.jsonl')

## 12. Parse batch results into standard synopsis records

This converts raw OpenAI batch outputs into a clean JSONL file with:

- `story_text_uid`
- `synopsis`
- status
- usage
- error fields

The parsed JSONL is append-only so the notebook can support multiple backfill batches.

In [37]:
def parse_openai_synopsis_batch_output(
    batch_output_path: Path,
    plan_csv_path: Path,
    parsed_jsonl_path: Path = SYNOPSIS_PARSED_JSONL,
) -> Path:
    """
    Parse OpenAI batch output into clean synopsis records.
    """
    plan_df = pd.read_csv(plan_csv_path)
    plan_by_custom_id = {
        row["custom_id"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    records = read_jsonl(batch_output_path)

    n_success = 0
    n_empty = 0
    n_error = 0

    for rec in records:
        custom_id = rec.get("custom_id")
        plan_row = plan_by_custom_id.get(custom_id, {})

        response = rec.get("response")
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            raw_text = extract_text_from_responses_api_body(body)
            synopsis = clean_synopsis_text(raw_text)
            usage = body.get("usage")

            if synopsis:
                status = "success"
                n_success += 1
                error_out = None
            else:
                status = "empty_text"
                n_empty += 1
                error_out = "No synopsis text extracted."

            out = {
                "story_text_uid": plan_row.get("story_text_uid"),
                "custom_id": custom_id,
                "status": status,
                "synopsis": synopsis if synopsis else None,
                "raw_synopsis": raw_text if raw_text else None,
                "story_word_count": plan_row.get("story_word_count"),
                "model": OPENAI_SYNOPSIS_MODEL,
                "created_at_utc": now_iso(),
                "provider_response_id": body.get("id"),
                "usage": usage,
                "error": error_out,
                "batch_output_file": str(batch_output_path),
            }
        else:
            n_error += 1
            out = {
                "story_text_uid": plan_row.get("story_text_uid"),
                "custom_id": custom_id,
                "status": "error",
                "synopsis": None,
                "raw_synopsis": None,
                "story_word_count": plan_row.get("story_word_count"),
                "model": OPENAI_SYNOPSIS_MODEL,
                "created_at_utc": now_iso(),
                "provider_response_id": None,
                "usage": None,
                "error": error,
                "batch_output_file": str(batch_output_path),
            }

        append_jsonl(parsed_jsonl_path, out)

    print("Parsed to:", parsed_jsonl_path)
    print("Success:", n_success)
    print("Empty:  ", n_empty)
    print("Error:  ", n_error)

    return parsed_jsonl_path


# Important: set this to the plan path corresponding to the batch you are parsing.
PLAN_CSV_FOR_BATCH = synopsis_batch_files["plan_csv_path"]

if BATCH_ID_TO_CHECK is None or synopsis_output_path is None:
    print("No completed batch output to parse.")
else:
    parsed_path = parse_openai_synopsis_batch_output(
        batch_output_path=synopsis_output_path,
        plan_csv_path=PLAN_CSV_FOR_BATCH,
        parsed_jsonl_path=SYNOPSIS_PARSED_JSONL,
    )

Parsed to: analysis_outputs/story_synopses/parsed/openai__gpt-5.4__story_plot_synopses.jsonl
Success: 9356
Empty:   0
Error:   0


## 13. Inspect synopsis quality

This is a lightweight quality check. We are not filtering by quality here, but we want to flag obvious problems:

- empty synopsis,
- very long synopsis,
- multiple-sentence synopsis,
- accidental labels.

In [38]:
synopsis_records = read_jsonl(SYNOPSIS_PARSED_JSONL)
synopsis_df = pd.DataFrame(synopsis_records)

print("Parsed synopsis records:", synopsis_df.shape)

if len(synopsis_df) > 0:
    # Keep the latest successful synopsis per story_text_uid if multiple batches were parsed.
    synopsis_df["is_success"] = synopsis_df["status"].eq("success") & synopsis_df["synopsis"].notna()

    synopsis_success_df = (
        synopsis_df
        .query("is_success")
        .drop_duplicates("story_text_uid", keep="last")
        .copy()
    )

    synopsis_success_df["synopsis_word_count"] = synopsis_success_df["synopsis"].map(rough_word_count)
    synopsis_success_df["synopsis_sentenceish_count"] = synopsis_success_df["synopsis"].map(
        lambda x: max(1, len(re.findall(r"[.!?]", x))) if isinstance(x, str) and x.strip() else 0
    )
    synopsis_success_df["has_label_prefix"] = synopsis_success_df["synopsis"].str.contains(
        r"^(synopsis|plot synopsis)\s*:",
        case=False,
        regex=True,
        na=False,
    )

    print("Unique successful story_text_uid:", synopsis_success_df["story_text_uid"].nunique())
    display(synopsis_success_df["synopsis_word_count"].describe(percentiles=[.25, .5, .75, .9, .95, .99]))
    display(synopsis_success_df["synopsis_sentenceish_count"].value_counts().sort_index())

    display(
        synopsis_success_df[
            ["story_text_uid", "synopsis", "synopsis_word_count", "synopsis_sentenceish_count", "has_label_prefix"]
        ].sample(n=min(20, len(synopsis_success_df)), random_state=42)
    )

    failures = synopsis_df.query("status != 'success'").copy()
    print("Failures/empty rows:", len(failures))
    display(failures.head(20))

Parsed synopsis records: (9356, 12)
Unique successful story_text_uid: 9356


/var/folders/rj/l30_wb7d3w7_tbx4gbz6lzzh0000gn/T/ipykernel_66311/1989332232.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  synopsis_success_df["has_label_prefix"] = synopsis_success_df["synopsis"].str.contains(


count    9356.000000
mean       33.896537
std         7.777421
min        14.000000
25%        28.000000
50%        33.000000
75%        39.000000
90%        45.000000
95%        49.000000
99%        55.000000
max        70.000000
Name: synopsis_word_count, dtype: float64

synopsis_sentenceish_count
1    9176
2     144
3      35
4       1
Name: count, dtype: int64

,story_text_uid,synopsis,synopsis_word_count,synopsis_sentenceish_count,has_label_prefix
4635,141fc9ec48ed7a9c1fc4d2e0,"A babysitter thinks she is watching a sleeping girl named Emma until a phone call reveals the family has only a son, and when she rushes upstairs she finds a monstrous impostor...",52,1,False
3395,56f65437fd9881ddb4f25bec,"As a skydiver’s main and reserve parachutes both fail during a jump, they realize they are about to die and face the imminent impact with grim acceptance.",28,1,False
8631,d0976390ccf3b28153ca196f,"After being persuaded by Greg to go skydiving, a narrator’s parachute fails to open during the jump, leaving them in a panic as they plummet helplessly toward the pointed trees...",32,1,False
6983,b6b72cd1dd85f143fd685db2,"A bitter, dying narrator spends their final moments in pain and contempt for the nurse and the world around them before slipping into death and darkness.",26,1,False
6611,7fe0c7f7105ff6a0e7b8c2f0,"A fatalistic narrator plummets toward the ground after their parachute fails, briefly reflecting on their pattern of disappointment before resigning themself to the likely fata...",26,1,False
8193,af77e5e1e612962c54847241,"In a silent old house, a reclusive narrator tries to preserve the isolated, shuttered life their mother preferred, but when a persistent mailman repeatedly knocks, the narrator...",40,1,False
624,99a2d57decb94b754e0bee5a,"At a party, a narrator investigating an uncanny duplicate laugh discovers their dead mother in the attic wearing their face and claiming they are the copy, just before the gues...",41,1,False
9318,20072a9f2609b600a197706e,"After being given a faulty parachute, a despondent skydiver falls toward the ground and resigns himself to dying because he believes no one cares about him.",26,1,False
3033,7832db7184811bca20607564,"At ninety-nine, the fiercely defiant Aurelia spends her final ten seconds in a hospital bed reliving flashes of her turbulent life, resisting fear and indignity until she confr...",37,1,False
4828,471d804c94b44265f152ca66,"After inheriting her mother's house, a woman finally opens the forbidden spare room, only to discover signs that something had been trapped inside, and when the door locks behi...",41,1,False


Failures/empty rows: 0


,story_text_uid,custom_id,status,synopsis,raw_synopsis,story_word_count,model,created_at_utc,provider_response_id,usage,error,batch_output_file,is_success


## 14. Merge synopses back onto all story rows

We now map each unique text-level synopsis back to every story row. This preserves duplicates and all metadata needed for downstream crowding analysis.

In [39]:
# Latest successful synopsis per unique story text.
synopsis_success_df = (
    synopsis_df
    .query("status == 'success' and synopsis.notna()")
    .drop_duplicates("story_text_uid", keep="last")
    .copy()
)

story_with_text_uid_df = story_all_df.merge(
    unique_story_text_df[["story_text", "story_text_uid"]],
    on="story_text",
    how="left",
    validate="many_to_one",
)

story_synopsis_merged_df = story_with_text_uid_df.merge(
    synopsis_success_df[
        [
            "story_text_uid",
            "synopsis",
            "raw_synopsis",
            "model",
            "provider_response_id",
            "usage",
        ]
    ].rename(columns={
        "model": "synopsis_model",
        "provider_response_id": "synopsis_response_id",
        "usage": "synopsis_usage",
    }),
    on="story_text_uid",
    how="left",
    validate="many_to_one",
)

story_synopsis_merged_df["has_synopsis"] = story_synopsis_merged_df["synopsis"].notna()

print("Merged story rows:", story_synopsis_merged_df.shape)
print("Rows with synopsis:", int(story_synopsis_merged_df["has_synopsis"].sum()))
print("Rows missing synopsis:", int((~story_synopsis_merged_df["has_synopsis"]).sum()))

display(
    story_synopsis_merged_df
    .groupby(["source_type", "provider", "model", "analysis_scenario_name"], dropna=False)
    .agg(
        n_rows=("story_uid", "size"),
        n_with_synopsis=("has_synopsis", "sum"),
        n_unique_story_texts=("story_text_uid", "nunique"),
    )
    .reset_index()
    .sort_values(["source_type", "provider", "analysis_scenario_name"])
)

missing_synopsis = story_synopsis_merged_df.query("not has_synopsis").copy()
display(missing_synopsis.head())

Merged story rows: (9357, 21)
Rows with synopsis: 9357
Rows missing synopsis: 0


,source_type,provider,model,analysis_scenario_name,n_rows,n_with_synopsis,n_unique_story_texts
0,ai,anthropic,claude-sonnet-4-5,neutral_main_t1,150,150,150
1,ai,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,60,60,60
2,ai,anthropic,claude-sonnet-4-5,personality_grid,2880,2880,2879
3,ai,gemini,gemini-2.5-flash,neutral_main_t1,150,150,150
4,ai,gemini,gemini-2.5-flash,neutral_temperature_robustness,60,60,60
5,ai,gemini,gemini-2.5-flash,personality_grid,2880,2880,2880
6,ai,openai,gpt-5.4,neutral_main_t1,150,150,150
7,ai,openai,gpt-5.4,neutral_temperature_robustness,60,60,60
8,ai,openai,gpt-5.4,personality_grid,2880,2880,2880
9,human,human,human,human,87,87,87


,source_type,provider,model,analysis_scenario_name,scenario_name,temperature,persona_id,condition_id,story_prompt,participant_id,story_text,story_row_idx,story_uid,story_word_count,story_text_uid,synopsis,raw_synopsis,synopsis_model,synopsis_response_id,synopsis_usage,has_synopsis


## 15. Save final synopsis files

We save:

1. the unique text-level synopsis table;
2. the full story-row-level merged table;
3. compact main-analysis-ready files.

The row-level merged file is the most important one for the downstream story narrative-crowding analysis.

In [40]:
# Unique text-level table.
unique_synopsis_df = unique_story_text_df.merge(
    synopsis_success_df[
        [
            "story_text_uid",
            "synopsis",
            "raw_synopsis",
            "model",
            "provider_response_id",
            "usage",
        ]
    ].rename(columns={
        "model": "synopsis_model",
        "provider_response_id": "synopsis_response_id",
        "usage": "synopsis_usage",
    }),
    on="story_text_uid",
    how="left",
    validate="one_to_one",
)

unique_synopsis_df["has_synopsis"] = unique_synopsis_df["synopsis"].notna()
unique_synopsis_df["synopsis_word_count"] = unique_synopsis_df["synopsis"].map(rough_word_count)

unique_text_pkl = FINAL_DIR / f"openai__{OPENAI_SYNOPSIS_MODEL}__unique_story_text_synopses.pkl"
unique_text_csv = FINAL_DIR / f"openai__{OPENAI_SYNOPSIS_MODEL}__unique_story_text_synopses.csv"

merged_pkl = FINAL_DIR / f"openai__{OPENAI_SYNOPSIS_MODEL}__story_plot_synopses_merged.pkl"
merged_csv = FINAL_DIR / f"openai__{OPENAI_SYNOPSIS_MODEL}__story_plot_synopses_merged.csv"

unique_synopsis_df.to_pickle(unique_text_pkl)
unique_synopsis_df.to_csv(unique_text_csv, index=False)

story_synopsis_merged_df.to_pickle(merged_pkl)
story_synopsis_merged_df.to_csv(merged_csv, index=False)

print("Saved:")
print(unique_text_pkl)
print(unique_text_csv)
print(merged_pkl)
print(merged_csv)

Saved:
analysis_outputs/story_synopses/final/openai__gpt-5.4__unique_story_text_synopses.pkl
analysis_outputs/story_synopses/final/openai__gpt-5.4__unique_story_text_synopses.csv
analysis_outputs/story_synopses/final/openai__gpt-5.4__story_plot_synopses_merged.pkl
analysis_outputs/story_synopses/final/openai__gpt-5.4__story_plot_synopses_merged.csv


In [41]:
print("=" * 100)
print("SYNOPSIS GENERATION STATUS")
print("=" * 100)

print("Unique story texts:", len(unique_synopsis_df))
print("Unique story texts with synopsis:", int(unique_synopsis_df["has_synopsis"].sum()))
print("Unique story texts missing synopsis:", int((~unique_synopsis_df["has_synopsis"]).sum()))

print("\nStory rows:", len(story_synopsis_merged_df))
print("Story rows with synopsis:", int(story_synopsis_merged_df["has_synopsis"].sum()))
print("Story rows missing synopsis:", int((~story_synopsis_merged_df["has_synopsis"]).sum()))

print("\nKey files for downstream analysis:")
print(merged_pkl)
print(merged_csv)
print(unique_text_pkl)
print(unique_text_csv)

display(
    story_synopsis_merged_df
    .groupby(["source_type", "provider", "model", "analysis_scenario_name"], dropna=False)
    .agg(
        n_rows=("story_uid", "size"),
        n_with_synopsis=("has_synopsis", "sum"),
        n_unique_story_texts=("story_text_uid", "nunique"),
    )
    .reset_index()
    .sort_values(["source_type", "provider", "analysis_scenario_name"])
)

SYNOPSIS GENERATION STATUS
Unique story texts: 9356
Unique story texts with synopsis: 9356
Unique story texts missing synopsis: 0

Story rows: 9357
Story rows with synopsis: 9357
Story rows missing synopsis: 0

Key files for downstream analysis:
analysis_outputs/story_synopses/final/openai__gpt-5.4__story_plot_synopses_merged.pkl
analysis_outputs/story_synopses/final/openai__gpt-5.4__story_plot_synopses_merged.csv
analysis_outputs/story_synopses/final/openai__gpt-5.4__unique_story_text_synopses.pkl
analysis_outputs/story_synopses/final/openai__gpt-5.4__unique_story_text_synopses.csv


,source_type,provider,model,analysis_scenario_name,n_rows,n_with_synopsis,n_unique_story_texts
0,ai,anthropic,claude-sonnet-4-5,neutral_main_t1,150,150,150
1,ai,anthropic,claude-sonnet-4-5,neutral_temperature_robustness,60,60,60
2,ai,anthropic,claude-sonnet-4-5,personality_grid,2880,2880,2879
3,ai,gemini,gemini-2.5-flash,neutral_main_t1,150,150,150
4,ai,gemini,gemini-2.5-flash,neutral_temperature_robustness,60,60,60
5,ai,gemini,gemini-2.5-flash,personality_grid,2880,2880,2880
6,ai,openai,gpt-5.4,neutral_main_t1,150,150,150
7,ai,openai,gpt-5.4,neutral_temperature_robustness,60,60,60
8,ai,openai,gpt-5.4,personality_grid,2880,2880,2880
9,human,human,human,human,87,87,87
